# VinFast ADAS: BEV/4D Occupancy & Flow Benchmarking Lab
**Mã dự án:** RAV-25 | **Target:** Kaggle GPU (T4 16GB) | **Ground Truth:** Chuẩn Occ3D-nuScenes

Pipeline hoàn chỉnh gồm:
1. **Cài đặt thư viện** xe tự hành & chuẩn bị môi trường.
2. **Nạp bộ nhãn chuẩn Occ3D Ground Truth** (Voxel $200 \times 200 \times 16$, 18 classes).
3. **Chạy Master Pipeline:** Nạp dữ liệu đa cảm biến (6 Camera + LiDAR + 5 Radar), Fine-tuning mô hình 4D-OccFusion với AMP FP16, đo đạc Benchmark đối đầu 4 cấu hình cảm biến.
4. **Hiển thị trực quan** bản đồ 3D BEV Map & Occupancy Flow kết quả ngay trong Notebook.
5. **Xuất & Kiểm tra Trọn bộ Artifact 4D** chuẩn công nghiệp theo đặc tả (NPY, JSON, PLY, GIF, YAML).

In [ ]:
# Cell 1: Cài đặt thư viện phụ trợ cần thiết (Không phụ thuộc nuscenes-devkit để tránh xung đột NumPy 2.x)
!pip install -q pyquaternion einops gdown

In [ ]:
# Cell 2: Tải và thiết lập bộ nhãn chuẩn Occ3D Ground Truth cho nuScenes-mini
# File ID chính thức từ Tsinghua MARS Lab & CVPR Challenge: 1-XIJRS_Uy0tC6Th6KlMIlrzw8JH939cp
!python tools/download_occ3d.py --target-dir data/occ3d_cam4d

In [ ]:
# Cell 3: Kiểm tra cấu hình GPU được cấp bởi Kaggle
import torch
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Model     : {torch.cuda.get_device_name(0)}")
    print(f"Total VRAM    : {torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB")

In [ ]:
# Cell 4: Chạy toàn bộ Master Pipeline từ A đến Z
!python kaggle_run_all.py

In [ ]:
# Cell 5: Hiển thị trực quan Bản đồ BEV 4D Occupancy kết quả ngay trong Notebook
from IPython.display import Image, display
import os

if os.path.exists('bev_output.png'):
    print("Hiển thị bản đồ BEV Map Snapshot (Xanh lá = Ground Truth Occ3D | Cam = Dự đoán | Đỏ = Nguy cơ va chạm):")
    display(Image('bev_output.png', width=700))
else:
    print("Chưa tìm thấy file bev_output.png, hãy chắc chắn Cell 4 đã chạy thành công!")

In [ ]:
# Cell 6: Khám phá Trọn bộ Gói Artifact 4D Occupancy (model_output/)
import os, json, pandas as pd
from IPython.display import Image, display

print("CẤU TRÚC GÓI ARTIFACT 4D MODEL OUTPUT:")
for root, dirs, files in os.walk('model_output'):
    level = root.replace('model_output', '').count(os.sep)
    indent = ' ' * 4 * level
    print(f"{indent}{os.path.basename(root)}/")
    sub_indent = ' ' * 4 * (level + 1)
    for f in files:
        sz = os.path.getsize(os.path.join(root, f))
        print(f"{sub_indent}├── {f} ({sz/1024:.1f} KB)")

if os.path.exists("model_output/evaluation/metrics.json"):
    print("\n--- TỔNG HỢP CHỈ SỐ ĐỊNH LƯỢNG (metrics.json) ---")
    with open("model_output/evaluation/metrics.json") as f:
        print(json.dumps(json.load(f), indent=2))

if os.path.exists("model_output/evaluation/per_class_metrics.csv"):
    print("\n--- BẢNG ĐÁNH GIÁ CHI TIẾT TỪNG CLASS (per_class_metrics.csv) ---")
    df = pd.read_csv("model_output/evaluation/per_class_metrics.csv")
    display(df)

if os.path.exists("model_output/visualization/temporal.gif"):
    print("\n--- HOẠT ẢNH CHUỖI THỜI GIAN 4D OCCUPANCY (temporal.gif) ---")
    display(Image("model_output/visualization/temporal.gif", width=450))